In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from boardgames_recsys.data.filtering import filter_df
from boardgames_recsys.models.collaborative_filtering import calc_distance_matrix, get_KNN
from boardgames_recsys.data.matrix import get_matrix_user_game

In [ ]:
folder = "database_cleaned"
avis_clean  = pd.read_csv(f"{folder}/avis_clean.csv", index_col=0)
jeux_clean  = pd.read_csv(f"{folder}/jeux_clean.csv", index_col=0)
users       = pd.read_csv(f"trictrac_database/users.csv", names=["Username", "User id"])



In [ ]:
# Filter for min_reviews for users & games
min_reviews = 5
rev_filter = filter_df(avis_clean, min_reviews)

In [ ]:
# generate user-game matrix
matrix_ratings, mask_ratings, users_table, games_table = get_matrix_user_game(rev_filter)

In [ ]:
# cosine similarity matrix, set k = sqrt(nb users)
cos_sim_matrix = calc_distance_matrix(matrix_ratings, mask_ratings, "cos")
k = int(np.sqrt(cos_sim_matrix.shape[0]))
cos_sim_matrix

Division of users by reviews
1. plus que 400
2. entre 100 et 400
3. de 6 à 100

In [ ]:
# Find users for who we will plot distances. Create dataframe : User index (in matrix), User id (in DB), Count reviews (per user)
#users_ids = rev_filter[["User id", "Game id"]].groupby("User id", as_index=True).count().sort_values("Game id", ascending=False).head(12)
users_ids =  rev_filter[["User id", "Game id"]].groupby("User id", as_index=True).count()
users_ids = users_ids[(users_ids["Game id"] >= 100)& (users_ids["Game id"] < 420)].sample(12)

assoc = users_table.to_frame().merge(users_ids, left_on="User id", right_index=True).reset_index()
assoc.columns = ["User index", "User id", "Count reviews"]
user_indices = assoc["User index"].to_numpy() # contains selected users indices

users_ids 

In [ ]:
# for each user -> find similar k users -> then find an array of distances (only for these k users)
users_knn = np.array([cos_sim_matrix[user, get_KNN(cos_sim_matrix, k=k, user_ind=user, dtype="cos")[:k]] for user in user_indices])
users_knn.shape

In [ ]:
user_distances = np.round(users_knn, 2) # eliminate distance precision 
dist_for_df  = user_distances.ravel()   # N-D to 1-D array
users_for_df = np.repeat(user_indices, k) # repeat each user's index k times (to construct dataframe in the next cell)
users_for_df.shape, dist_for_df.shape

In [ ]:
# Constuct Dataframe : where to each user index we associate an array of distances of k nearest users 
# e.g. if for user 0 we get k=3 similar users with distances [d1, d2, d3] then the dataframe will contain
# User index | Distance
# 0          | d1
# 0          | d2
# 0          | d3
distance_to_users = pd.DataFrame({"Distance": dist_for_df, "User index":users_for_df}).sort_values("Distance")

# Group by User index, distance (precision already eliminated) and apply .size(), i.e.
# for each user X we count how many similar users have the distance d.dd to the user X
distance_to_users = distance_to_users.groupby(["Distance", "User index"], as_index=True).size().reset_index(name="Number of users")
distance_to_users

In [ ]:
sns.set_theme(rc={'figure.figsize':(14,6)})

# Grid with 4 columns
g = sns.FacetGrid(distance_to_users, col="User index", col_wrap=4, xlim=(0.4, 1.))

# sns.scatterplot, "Distance", "Number of users" will be on each cell in a grid
g.map(sns.scatterplot, "Distance", "Number of users") 
g.figure.subplots_adjust(top=0.9) # allocate space for title 
g.figure.suptitle("Users [100-420 reviews]. k = 54 cos")